In [0]:
from pyspark.sql.functions import (trim,col,when,to_date,lower,to_timestamp)
patient_df = spark.table("workspace.silver.patient")
encounter_df = spark.table("workspace.silver.encounter")
observation_df = spark.table("workspace.silver.observation")
condition_df = spark.table("workspace.silver.condition")



In [0]:
patient_df = (

    patient_df

    # Trim strings
    .withColumn("patient_id", trim(col("patient_id")))
    .withColumn("gender", trim(col("gender")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn("country", trim(col("country")))

    # Empty string → NULL
    .withColumn(
        "city",
        when(col("city") == "", None).otherwise(col("city"))
    )

    .withColumn(
        "state",
        when(col("state") == "", None).otherwise(col("state"))
    )

    .withColumn(
        "country",
        when(col("country") == "", None).otherwise(col("country"))
    )

    # Standardize Gender
    .withColumn(
        "gender",
        when(lower(col("gender")) == "male", "Male")
        .when(lower(col("gender")) == "female", "Female")
        .otherwise("Unknown")
    )

    # Convert Date
    .withColumn(
        "birth_date",
        to_date(col("birth_date"))
    )

    # Boolean
    .withColumn(
        "active",
        when(lower(col("active")) == "true", True)
        .otherwise(False)
    )

    # Remove invalid business key
    .filter(col("patient_id").isNotNull())

    # Remove duplicates
    .dropDuplicates(["patient_id"])

)

patient_df.display()

In [0]:
encounter_df = (

    encounter_df

    .withColumn("encounter_id", trim(col("encounter_id")))
    .withColumn("status", trim(col("status")))
    .withColumn("class", trim(col("class")))

    .withColumn(
        "status",
        when(col("status") == "", None).otherwise(col("status"))
    )

    .withColumn(
        "start_date",
        to_date(col("start_date"))
    )

    .withColumn(
        "end_date",
        to_date(col("end_date"))
    )

    .filter(col("encounter_id").isNotNull())

    .dropDuplicates(["encounter_id"])

)
encounter_df.display()

In [0]:
observation_df = (

    observation_df

    .withColumn(
        "observation_id",
        trim(col("observation_id"))
    )

    .withColumn(
        "status",
        trim(col("status"))
    )

    .withColumn(
        "observation_code",
        trim(col("observation_code"))
    )

    .withColumn(
        "effective_datetime",
        to_timestamp(col("effective_datetime"))
    )

    .filter(col("observation_id").isNotNull())

    .dropDuplicates(["observation_id"])

)
observation_df.display()

In [0]:
condition_df = (

    condition_df

    .withColumn(
        "condition_id",
        trim(col("condition_id"))
    )

    .withColumn(
        "clinical_status",
        trim(col("clinical_status"))
    )

    .withColumn(
        "verification_status",
        trim(col("verification_status"))
    )

    .withColumn(
        "condition",
        trim(col("condition"))
    )

    .withColumn(
        "onset_datetime",
        to_timestamp(col("onset_datetime"))
    )

    .filter(col("condition_id").isNotNull())

    .dropDuplicates(["condition_id"])

)

condition_df.display()

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    coalesce,
    current_timestamp,
    lit,
    regexp_replace
)

# =====================================================
# Read Silver Tables
# =====================================================

patient_df = spark.table(
    "workspace.silver.patient"
)

encounter_df = spark.table(
    "workspace.silver.encounter"
)

observation_df = spark.table(
    "workspace.silver.observation"
)

condition_df = spark.table(
    "workspace.silver.condition"
)

# =====================================================
# Build Encounter Metrics
# =====================================================

encounter_metrics = (

    encounter_df

    .withColumn(
        "patient_id",
        regexp_replace(
            col("patient_reference"),
            "Patient/",
            ""
        )
    )

    .groupBy("patient_id")

    .agg(
        count("*").alias("encounter_count")
    )

)

# =====================================================
# Build Observation Metrics
# =====================================================

observation_metrics = (

    observation_df

    .withColumn(
        "patient_id",
        regexp_replace(
            col("patient_reference"),
            "Patient/",
            ""
        )
    )

    .groupBy("patient_id")

    .agg(
        count("*").alias("observation_count")
    )

)

# =====================================================
# Build Condition Metrics
# =====================================================

condition_metrics = (

    condition_df

    .withColumn(
        "patient_id",
        regexp_replace(
            col("patient_reference"),
            "Patient/",
            ""
        )
    )

    .groupBy("patient_id")

    .agg(
        count("*").alias("condition_count")
    )

)

# =====================================================
# Build Gold Patient Summary
# =====================================================

gold_df = (

    patient_df.alias("p")

    .join(
        encounter_metrics.alias("e"),
        "patient_id",
        "left"
    )

    .join(
        observation_metrics.alias("o"),
        "patient_id",
        "left"
    )

    .join(
        condition_metrics.alias("c"),
        "patient_id",
        "left"
    )

    .select(

        col("patient_id"),

        col("gender"),

        col("birth_date"),

        col("address"),


        coalesce(
            col("encounter_count"),
            lit(0)
        ).alias("encounter_count"),

        coalesce(
            col("observation_count"),
            lit(0)
        ).alias("observation_count"),

        coalesce(
            col("condition_count"),
            lit(0)
        ).alias("condition_count"),

        current_timestamp().alias("load_timestamp")

    )

)

# =====================================================
# Display Gold Data
# =====================================================

display(gold_df)

# =====================================================
# Write Gold Table
# =====================================================

# (
#     gold_df.write

#     .format("delta")

#     .mode("overwrite")

#     .saveAsTable(
#         "workspace.default.gold_patient_summary"
#     )

# )

print("Gold Patient Summary Created Successfully")